In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ============================================================
# BlockD — Quick Look: Vertical Profile (Manual Time Axis)
# ============================================================

# ---------------- config ----------------
IN_FILE = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EHF_BWCN002_vTprime_ALL_LAT_stdplev_time_lat.nc"
OUT_FIG = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/Fig_BlockD_EHF_Profile_0007-0008.png"

# ---------------- main ----------------
# 1. 读取数据
print(f"[BlockD] Reading: {IN_FILE}")
ds = xr.open_dataset(IN_FILE)

# 2. 单位转换 Pa -> hPa
ds["plev"] = ds["plev"] / 100.0
ds["plev"].attrs["units"] = "hPa"

# 3. 切片 (0007-11 ~ 0008-06, 40-80N, 1-100hPa)
# 虽然绘图报错，但 xarray 的 sel(time=...) 切片功能通常是正常的
print("[BlockD] Slicing data...")
subset = ds["EHF_vTprime_stdplev"].sel(
    time=slice("0007-11-01", "0008-06-30"),
    lat=slice(40, 80),
    plev=slice(1, 100) # 注意：如果数据是降序(100->1)，这里可能需要反过来写，或者xarray自动处理
)

# 4. 区域平均
data_plot = subset.mean(dim="lat")  # (time, plev)

# ==========================================================
# 核心修改：构建纯数字的 X 轴，绕过 cftime 问题
# ==========================================================
print("[BlockD] Preparing manual time axis...")

# 1. 生成 X 轴的数字坐标 (0, 1, 2, ...)
x_indices = np.arange(len(subset.time))

# 2. 生成对应的标签字符串 (用于 X 轴显示)
# cftime 对象通常有 strftime 方法
time_labels = [t.strftime("%Y-%m") for t in subset.time.values]

# 3. 转置数据以适应 contourf (通常习惯 X=Time, Y=Lev)
# xarray plot 自动处理转置，但用 matplotlib 原生命令最好手动确认维度
# data_plot.values 的 shape 应该是 (time, plev)
# 我们需要传入 (X, Y, Z)
X, Y = np.meshgrid(x_indices, data_plot.plev.values)
Z = data_plot.values.T  # 转置成 (plev, time) 以匹配 meshgrid

# ==========================================================
# 绘图
# ==========================================================
print("[BlockD] Plotting...")
fig, ax = plt.subplots(figsize=(12, 6))

# 使用 matplotlib 原生 contourf，不通过 xarray 调用
p = ax.contourf(
    X, Y, Z,
    levels=20,
    cmap="RdBu_r",
    extend="both"
)

# ---- 调整 Y 轴 (对数 + 反转) ----
ax.set_yscale("log")
ax.set_ylim(100, 1)  # 底部 100，顶部 1
ax.set_ylabel("Pressure (hPa)")
# 手动设置 Y 轴刻度文本
from matplotlib.ticker import ScalarFormatter
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.set_yticks([1, 2, 5, 10, 20, 50, 100])

# ---- 调整 X 轴 (手动设置标签) ----
ax.set_xlabel("Time")

# 为了不让 X 轴太拥挤，我们每隔 30 天(大约1个月)显示一个标签
# 或者简单点，显示 8 个刻度
n_ticks = 10
tick_locs = np.linspace(0, len(x_indices)-1, n_ticks, dtype=int)
tick_labels = [time_labels[i] for i in tick_locs]

ax.set_xticks(tick_locs)
ax.set_xticklabels(tick_labels, rotation=30, ha="right")

# ---- 标题与 Colorbar ----
ax.set_title("Eddy Heat Flux (v'T') Vertical Profile [40-80N Mean]\nPeriod: 0007-11 to 0008-06")
cbar = plt.colorbar(p, ax=ax, label="K m s$^{-1}$")

plt.tight_layout()

# 6. 保存
print(f"[BlockD] Saving figure to: {OUT_FIG}")
plt.savefig(OUT_FIG, dpi=300)
# plt.show() 

print("[BlockD] Done.")

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ============================================================
# BlockD — Standardized Anomaly of EHF Vertical Profile
# Method: (Val - ClimMean) / ClimStd
# Focus: 40-80N Mean, 100-1 hPa, Target: 0007-11 to 0008-06
# ============================================================

# ---------------- config ----------------
IN_FILE = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EHF_BWCN002_vTprime_ALL_LAT_stdplev_time_lat.nc"
OUT_FIG = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/Fig_BlockD_EHF_StdAnomaly_0007-0008.png"

# ---------------- main ----------------
print(f"[BlockD] Reading: {IN_FILE}")
ds = xr.open_dataset(IN_FILE)

# 1. 单位转换 Pa -> hPa
ds["plev"] = ds["plev"] / 100.0
ds["plev"].attrs["units"] = "hPa"

# 2. 先做空间平均 (40-80N)
# 注意：我们使用整个时间段的数据来计算气候态，所以这里只切 Lat，不切 Time
print("[BlockD] Computing 40-80N mean for the full period...")
data_lat_mean = ds["EHF_vTprime_stdplev"].sel(lat=slice(40, 80)).mean(dim="lat")
# 此时 data_lat_mean 形状为 (All_Time, plev)

# 3. 计算标准化距平 (Standardized Anomaly)
print("[BlockD] Calculating Climatology (Daily Mean & Std) and Anomaly...")

# 使用 groupby('time.dayofyear') 按“年积日”进行分组
# 这样会将所有年份的第1天放在一组，第2天放在一组...
clim_mean = data_lat_mean.groupby("time.dayofyear").mean("time")
clim_std  = data_lat_mean.groupby("time.dayofyear").std("time")

# 计算距平: (原始值 - 气候均值) / 气候标准差
# xarray 会自动处理广播 (broadcasting)，把 climatology 匹配回对应的时间点
# 注意：如果 clim_std 极小，可能会出现数值不稳定，但在 EHF 场中一般没事
std_anomaly = xr.apply_ufunc(
    lambda x, m, s: (x - m) / s,
    data_lat_mean.groupby("time.dayofyear"),
    clim_mean,
    clim_std
)

# 移除 groupby 产生的 dayofyear 坐标信息，还原回纯 time 坐标（如果需要的话）
# xarray 计算完通常会保留原坐标，这里通常不需要额外操作

# 4. 选取目标时间段 & 高度层
print("[BlockD] Slicing target period (0007-11 to 0008-06)...")
subset_plot = std_anomaly.sel(
    time=slice("0007-11-01", "0008-06-30"),
    plev=slice(1, 100) # 100hPa 到 1hPa
)

# ==========================================================
# 绘图 (Robust Method - 纯数字坐标)
# ==========================================================
print("[BlockD] Plotting Standardized Anomaly...")

# 准备绘图数据
X_idx = np.arange(len(subset_plot.time))
Y_val = subset_plot.plev.values
Z_val = subset_plot.values.T  # (plev, time)

# 准备时间标签
time_labels = [t.strftime("%Y-%m") for t in subset_plot.time.values]

fig, ax = plt.subplots(figsize=(12, 6))

# 设置等值线 levels
# 标准化距平通常在 -3 到 +3 之间 (Sigma)
levels = np.linspace(-4, 4, 21) # 每隔 0.4 一个色阶

p = ax.contourf(
    X_idx, Y_val, Z_val,
    levels=levels,
    cmap="RdBu_r", # 红色正距平(偏强)，蓝色负距平(偏弱)
    extend="both"
)

# ---- Y 轴设置 ----
ax.set_yscale("log")
ax.set_ylim(100, 1)
ax.set_ylabel("Pressure (hPa)")
from matplotlib.ticker import ScalarFormatter
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.set_yticks([1, 2, 5, 10, 20, 50, 100])

# ---- X 轴设置 ----
ax.set_xlabel("Time")
n_ticks = 10
tick_locs = np.linspace(0, len(X_idx)-1, n_ticks, dtype=int)
tick_labels = [time_labels[i] for i in tick_locs]
ax.set_xticks(tick_locs)
ax.set_xticklabels(tick_labels, rotation=30, ha="right")

# ---- 标题与 Colorbar ----
ax.set_title("Standardized Anomaly of Eddy Heat Flux (v'T') [40-80N Mean]\n"
             "Reference: Daily Climatology over Full Period | Unit: Sigma ($\sigma$)")

# Colorbar 加上 ticks 以便看清 sigma 值
cbar = plt.colorbar(p, ax=ax, ticks=[-4, -3, -2, -1, 0, 1, 2, 3, 4], label="Standard Deviation ($\sigma$)")

plt.tight_layout()

print(f"[BlockD] Saving figure to: {OUT_FIG}")
plt.savefig(OUT_FIG, dpi=300)
# plt.show()

print("[BlockD] Done.")